# Notebook 5 — V-JEPA2 Verb Classification on YouCook2

This notebook trains a classification head on top of a **frozen V-JEPA2 encoder** to predict masked verbs from YouCook2 video segments.

**Design rationale:**
- The V-JEPA2 backbone is frozen — only the classification head trains
- Unlike NB4 (DinoTXT + mean pooling), V-JEPA2 natively models temporal structure
- Each segment is trimmed to its `start_sec`/`end_sec` boundaries before being
  passed to the processor — no manual mean pooling needed
- The processor handles resizing and normalization internally

**Reads:** `annotations/segments_verbs_timed.csv`

**Model:** `facebook/vjepa2-vitl-fpc16-256-ssv2` — pretrained V-JEPA2 ViT-L
with SSv2 classification head replaced by a fresh verb head

## Install dependencies

In [1]:
!pip install -q transformers accelerate
!pip install -q decord

## Setup

In [2]:
import os, json
import torch
import pandas as pd
import numpy as np

with open('/home/ubuntu/youcook2-videos/paths.json') as f:
    PATHS = json.load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

Device : cuda
GPU    : NVIDIA A10


## Load data

In [13]:
df = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs_timed_clean.csv')

# Build verb vocabulary from extracted verbs
verb_vocab  = sorted(df['extracted_verb'].unique())
label2id    = {v: i for i, v in enumerate(verb_vocab)}
id2label    = {i: v for v, i in label2id.items()}
n_verbs     = len(verb_vocab)

df['verb_label'] = df['extracted_verb'].map(label2id)

print(f"Total segments : {len(df):,}")
print(f"Verb classes   : {n_verbs}")
print(f"Columns        : {df.columns.tolist()}")
print(df.head(2))

Total segments : 13,003
Verb classes   : 282
Columns        : ['caption', 'video_id', 'segment_id', 'extracted_verb', 'masked_caption', 'start_sec', 'end_sec', 'subset', 'verb_label']
                             caption     video_id       segment_id  \
0          crush and chop the garlic  --bv0V6ZjWI  --bv0V6ZjWI_000   
1  add oil garlic and salt to a bowl  --bv0V6ZjWI  --bv0V6ZjWI_001   

  extracted_verb                        masked_caption  start_sec  end_sec  \
0          crush            [MASK] and chop the garlic        8.0     23.0   
1            add  [MASK] oil garlic and salt to a bowl       23.0     47.0   

     subset  verb_label  
0  training          56  
1  training           1  


## Train / Val split

YouCook2 provides official train/val subsets via the `subset` column.

In [4]:
df_train = df[df['subset'] == 'training'].reset_index(drop=True)
df_val   = df[df['subset'] == 'validation'].reset_index(drop=True)

print(f"Train : {len(df_train):,}")
print(f"Val   : {len(df_val):,}")

Train : 9,735
Val   : 3,268


## Load V-JEPA2 model

Load processor and model first so we can read `frames_per_clip` from
the model config before defining the dataset.

`ignore_mismatched_sizes=True` replaces the pretrained SSv2 head
(174 classes) with a fresh head sized to our verb vocabulary.
The encoder weights are kept and then frozen.

In [9]:
for name, param in model.named_parameters():
    if "classifier" not in name and param.requires_grad:
        print(f"WARNING: {name} is not frozen!")

In [12]:
print(df.columns.tolist())
print(df.head(2))

['segment_id', 'subset', 'ground_truth', 'top1_pred', 'top5_preds', 'top1_correct', 'top5_correct', 'masked_caption']
        segment_id    subset ground_truth top1_pred  \
0  --bv0V6ZjWI_000  training        crush    sprout   
1  --bv0V6ZjWI_001  training          add  scramble   

                              top5_preds  top1_correct  top5_correct  \
0    [sprout, stem, tuck, stack, arange]         False         False   
1  [scramble, rotate, unil, grate, tbsp]         False         False   

                         masked_caption  
0            [MASK] and chop the garlic  
1  [MASK] oil garlic and salt to a bowl  


In [15]:
from transformers import VJEPA2ForVideoClassification, VJEPA2VideoProcessor

MODEL_NAME = 'facebook/vjepa2-vitl-fpc16-256-ssv2'

df = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs_timed_clean.csv')
verb_vocab   = sorted(df['extracted_verb'].unique())
label2id     = {v: i for i, v in enumerate(verb_vocab)}
id2label     = {i: v for v, i in label2id.items()}
num_classes  = len(verb_vocab)
df['verb_label'] = df['extracted_verb'].map(label2id)

assert num_classes == 282, f"Expected 282 verb classes, got {num_classes}"
print(f"Verb classes   : {num_classes}")

print(f"Loading {MODEL_NAME}...")
processor = VJEPA2VideoProcessor.from_pretrained(MODEL_NAME)
model     = VJEPA2ForVideoClassification.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    num_labels=num_classes,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
).to(DEVICE)

# Freeze only the ViT backbone — matches Meta's official fine-tuning approach
for param in model.vjepa2.parameters():
    param.requires_grad = False

FRAMES_PER_CLIP = model.config.frames_per_clip

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Total parameters     : {total/1e6:.1f}M")
print(f"Trainable parameters : {trainable/1e6:.2f}M  (pooler + classifier)")
print(f"Frames per clip      : {FRAMES_PER_CLIP}")
print(f"Num verb classes     : {model.config.num_labels}")

[transformers] You passed `num_labels=282` which is incompatible to the `id2label` map of length `174`.


Verb classes   : 282
Loading facebook/vjepa2-vitl-fpc16-256-ssv2...


Loading weights:   0%|          | 0/652 [00:00<?, ?it/s]

[transformers] VJEPA2ForVideoClassification LOAD REPORT from: facebook/vjepa2-vitl-fpc16-256-ssv2
Key               | Status   |                                                                                             
------------------+----------+---------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([174, 1024]) vs model:torch.Size([282, 1024])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([174]) vs model:torch.Size([282])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Total parameters     : 375.6M
Trainable parameters : 49.63M  (pooler + classifier)
Frames per clip      : 16
Num verb classes     : 282


## Dataset

Each segment is trimmed to its `start_sec`/`end_sec` boundaries.
`FRAMES_PER_CLIP` frames are sampled evenly within that window —
bounding `np.linspace` to `[start_idx, end_idx]` IS the trim.
We never sample outside the caption's temporal window.

The processor handles resizing and normalization. Unlike NB4 which
mean-pooled frame embeddings manually, the clip is passed directly
to V-JEPA2 which models temporal structure natively.

In [16]:
from torch.utils.data import Dataset, DataLoader
from decord import VideoReader, cpu

class YouCookVerbDataset(Dataset):
    def __init__(self, df, video_dir, frames_per_clip):
        self.df              = df.reset_index(drop=True)
        self.video_dir       = video_dir
        self.frames_per_clip = frames_per_clip

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        vid_path = os.path.join(self.video_dir, f"{row['video_id']}.mp4")

        vr        = VideoReader(vid_path, ctx=cpu(0))
        fps       = vr.get_avg_fps()

        # trim to caption boundaries
        start_idx = max(0, int(row['start_sec'] * fps))
        end_idx   = min(len(vr) - 1, int(row['end_sec'] * fps))

        # sample frames_per_clip evenly within the trimmed segment
        # linspace bounded to [start_idx, end_idx] IS the trim
        indices = np.linspace(start_idx, end_idx, self.frames_per_clip, dtype=int)

        # (T, H, W, 3) -> (T, C, H, W)
        frames = vr.get_batch(indices).asnumpy()
        clip   = torch.from_numpy(frames).permute(0, 3, 1, 2)

        return clip, int(row['verb_label'])


train_ds = YouCookVerbDataset(df_train, PATHS['videos'], FRAMES_PER_CLIP)
val_ds   = YouCookVerbDataset(df_val,   PATHS['videos'], FRAMES_PER_CLIP)

print(f"Train dataset : {len(train_ds):,} segments")
print(f"Val dataset   : {len(val_ds):,} segments")

# Sanity check
clip, label = train_ds[0]
print(f"Sample clip shape : {clip.shape}  (expected: ({FRAMES_PER_CLIP}, C, H, W))")
print(f"Sample label      : {label} -> {id2label[label]}")

Train dataset : 9,735 segments
Val dataset   : 3,268 segments
Sample clip shape : torch.Size([16, 3, 720, 1280])  (expected: (16, C, H, W))
Sample label      : 56 -> crush


## DataLoaders

The collate function passes clips through the V-JEPA2 processor
which handles resizing and normalization.

In [17]:
def collate_fn(batch):
    clips, labels = zip(*batch)
    # processor expects list of (T, C, H, W) tensors
    # handles resize, normalize, and returns pt tensors
    inputs = processor(list(clips), return_tensors='pt')
    return inputs, torch.tensor(labels, dtype=torch.long)


BATCH_SIZE  = 4
NUM_WORKERS = 4

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")

Train batches : 2,434
Val batches   : 817


## Training

- Backbone: **frozen** — no parameters update
- Classification head: **trainable** — attentive pooling + linear layer
- Loss: CrossEntropy on verb labels
- Gradient accumulation to simulate larger batch sizes on limited GPU memory

In [19]:
from tqdm.notebook import tqdm

NUM_EPOCHS           = 5
LR                   = 1e-5   # match official notebook
GRADIENT_ACCUM_STEPS = 4

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer        = torch.optim.Adam(trainable_params, lr=LR)
scheduler        = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.95)


def evaluate_top5(loader):
    model.eval()
    correct_top1, correct_top5, total = 0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            labels = labels.to(DEVICE)
            logits = model(**inputs).logits
            top5   = logits.topk(5, dim=-1).indices
            correct_top1 += (logits.argmax(-1) == labels).sum().item()
            correct_top5 += sum(
                labels[i].item() in top5[i].tolist()
                for i in range(len(labels))
            )
            total += labels.size(0)
    return correct_top1 / total, correct_top5 / total


CKPT_DIR = PATHS.get('checkpoints', '/home/ubuntu/youcook2-videos/checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"Starting training — {NUM_EPOCHS} epochs, LR={LR}, "
      f"batch={BATCH_SIZE}x{GRADIENT_ACCUM_STEPS} (accum)")
print(f"Trainable params: {sum(p.numel() for p in trainable_params)/1e6:.2f}M\n")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    running_loss = 0.0  # reset each epoch

    for step, (inputs, labels) in enumerate(
        tqdm(train_loader, desc=f"Epoch {epoch}"), start=1
    ):
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        labels = labels.to(DEVICE)

        outputs = model(**inputs, labels=labels)
        loss    = outputs.loss / GRADIENT_ACCUM_STEPS
        loss.backward()
        running_loss += outputs.loss.item()  # track unscaled loss

        if step % GRADIENT_ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()

    # flush leftover gradients if steps not divisible by accum steps
    if step % GRADIENT_ACCUM_STEPS != 0:
        optimizer.step()
        optimizer.zero_grad()

    scheduler.step()

    top1, top5 = evaluate_top5(val_loader)
    avg_loss = running_loss / step
    print(f"Epoch {epoch} | Avg Loss: {avg_loss:.4f} "
          f"| Val Top-1: {top1:.4f} | Val Top-5: {top5:.4f}")

    ckpt_path = os.path.join(CKPT_DIR, f'vjepa2_verb_epoch{epoch}.pt')
    torch.save(model.state_dict(), ckpt_path)
    print(f"Checkpoint saved -> {ckpt_path}")

Starting training — 5 epochs, LR=1e-05, batch=4x4 (accum)
Trainable params: 49.63M



Epoch 1:   0%|          | 0/2434 [00:00<?, ?it/s]

[h264 @ 0x275ef840] mmco: unref short failure
[h264 @ 0x275ef840] mmco: unref short failure
[h264 @ 0x27551700] mmco: unref short failure
[h264 @ 0x27447e00] mmco: unref short failure
[h264 @ 0x275eb840] mmco: unref short failure
[h264 @ 0x275eb840] mmco: unref short failure
[h264 @ 0x274271c0] mmco: unref short failure
[h264 @ 0x275ed380] mmco: unref short failure
[h264 @ 0x275ed380] mmco: unref short failure
[h264 @ 0x275ed380] mmco: unref short failure
[h264 @ 0x275eb840] mmco: unref short failure
[h264 @ 0x276206c0] mmco: unref short failure
[h264 @ 0x28cd6ac0] mmco: unref short failure
[h264 @ 0x28cd6ac0] mmco: unref short failure
[h264 @ 0x275ef840] mmco: unref short failure
[h264 @ 0x274271c0] mmco: unref short failure
[h264 @ 0x291cb980] mmco: unref short failure
[h264 @ 0x291cb980] mmco: unref short failure
[h264 @ 0x291cb980] mmco: unref short failure
[h264 @ 0x274271c0] mmco: unref short failure
[h264 @ 0x2c4176c0] mmco: unref short failure
[h264 @ 0x2753cac0] mmco: unref sh

Epoch 1 | Avg Loss: 3.3707 | Val Top-1: 0.3996 | Val Top-5: 0.6163
Checkpoint saved -> /home/ubuntu/youcook2-videos/checkpoints/vjepa2_verb_epoch1.pt


Epoch 2:   0%|          | 0/2434 [00:00<?, ?it/s]

[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x27536cc0] mmco: unref short failure
[h264 @ 0x27536cc0] mmco: unref short failure
[h264 @ 0x27536cc0] mmco: unref short failure
[h264 @ 0x27536700] mmco: unref short failure
[h264 @ 0x275699c0] mmco: unref short failure
[h264 @ 0x27536700] mmco: unref short failure
[h264 @ 0x27536700] mmco: unref short failure
[h264 @ 0x27536700] mmco: unref short failure
[h264 @ 0x275ed380] mmco: unref short failure
[h264 @ 0x275ed380] mmco: unref short failure
[h264 @ 0x25830b80] mmco: unref short failure
[h264 @ 0x25830b80] mmco: unref short failure
[h264 @ 0x275699c0] mmco: unref short failure
[h264 @ 0x275699c0] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x275699c0] mmco: unref short failure
[h264 @ 0x275699c0] mmco: unref sh

... (6 outputs left out)

Epoch 4 | Avg Loss: 2.4401 | Val Top-1: 0.4250 | Val Top-5: 0.7072
Checkpoint saved -> /home/ubuntu/youcook2-videos/checkpoints/vjepa2_verb_epoch4.pt


Epoch 5:   0%|          | 0/2434 [00:00<?, ?it/s]

[h264 @ 0x273c4c40] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x4ffd4d00] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x273c4c40] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref sh

Epoch 5 | Avg Loss: 2.3329 | Val Top-1: 0.4287 | Val Top-5: 0.7188
Checkpoint saved -> /home/ubuntu/youcook2-videos/checkpoints/vjepa2_verb_epoch5.pt


## Final evaluation

In [26]:
model.eval()
records = []
MAX_BATCHES = 50  # ~200 segments, runs in ~1 min

with torch.no_grad():
    for batch_idx, (inputs, labels) in enumerate(tqdm(val_loader, desc="Detailed eval")):
        if batch_idx >= MAX_BATCHES:
            break
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        labels = labels.to(DEVICE)
        logits = model(**inputs).logits
        top5   = logits.topk(5, dim=-1).indices

        for i in range(len(labels)):
            true_label = labels[i].item()
            top5_list  = top5[i].tolist()
            records.append({
                'video_id'    : df_val.iloc[len(records)]['video_id'],
                'segment_id'  : df_val.iloc[len(records)]['segment_id'],
                'ground_truth': id2label[true_label],
                'top1_pred'   : id2label[logits[i].argmax(-1).item()],
                'top1_correct': logits[i].argmax(-1).item() == true_label,
                'top5_correct': true_label in top5_list,
            })

results_df = pd.DataFrame(records)
wrong = results_df[results_df['top1_correct'] == False]
print(wrong['video_id'].value_counts().head(10))

Detailed eval:   0%|          | 0/817 [00:00<?, ?it/s]

[h264 @ 0x273baf80] mmco: unref short failure
[h264 @ 0x273baf80] mmco: unref short failure
[h264 @ 0x273c4c40] mmco: unref short failure
[h264 @ 0x273c4c40] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x501f0080] mmco: unref short failure
[h264 @ 0x51f0dcc0] mmco: unref short failure
[h264 @ 0x273c4c40] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x5175ad80] mmco: unref short failure
[h264 @ 0x5175ad80] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x455f4a80] mmco: unref short failure
[h264 @ 0x455f4a80] mmco: unref short failure
[h264 @ 0x455f4a80] mmco: unref short failure
[h264 @ 0x56a75100] mmco: unref sh

video_id
10dZTHlkb8w    8
0hb6NShH9hY    7
0uaKitJaqmI    7
2-mxsib6pJo    7
29Wkj1LqaK8    7
2IcWR76i1bo    7
-ErPSunMfcs    6
-GlSSp5ZOCQ    5
-Ju39A-G0Dk    5
-k7trpuj3X8    5
Name: count, dtype: int64


In [27]:
# Check if error rate is elevated for these videos vs their total segments
top_error_videos = wrong['video_id'].value_counts().head(10).index.tolist()

for vid in top_error_videos:
    total_in_batch  = results_df[results_df['video_id'] == vid].shape[0]
    errors_in_batch = wrong[wrong['video_id'] == vid].shape[0]
    error_rate      = errors_in_batch / total_in_batch if total_in_batch > 0 else 0
    print(f"{vid:<20} errors: {errors_in_batch}/{total_in_batch}  ({error_rate:.0%} error rate)")

10dZTHlkb8w          errors: 8/8  (100% error rate)
0hb6NShH9hY          errors: 7/11  (64% error rate)
0uaKitJaqmI          errors: 7/10  (70% error rate)
2-mxsib6pJo          errors: 7/14  (50% error rate)
29Wkj1LqaK8          errors: 7/9  (78% error rate)
2IcWR76i1bo          errors: 7/12  (58% error rate)
-ErPSunMfcs          errors: 6/7  (86% error rate)
-GlSSp5ZOCQ          errors: 5/9  (56% error rate)
-Ju39A-G0Dk          errors: 5/10  (50% error rate)
-k7trpuj3X8          errors: 5/6  (83% error rate)


In [28]:
# Inspect the specific failing segments for the suspicious videos
suspicious = ['10dZTHlkb8w', '-ErPSunMfcs', '-k7trpuj3X8']

for vid in suspicious:
    print(f"\n{'='*70}")
    print(f"Video: {vid}")
    print(f"{'='*70}")
    vid_rows = results_df[results_df['video_id'] == vid]
    for _, r in vid_rows.iterrows():
        correct = '✓' if r['top1_correct'] else '✗'
        # look up the caption for this segment
        caption_row = df_val[df_val['segment_id'] == r['segment_id']]
        caption = caption_row['masked_caption'].values[0] if len(caption_row) > 0 else 'N/A'
        print(f"  {correct} True: {r['ground_truth']:<12} Pred: {r['top1_pred']:<12} | {caption}")


Video: 10dZTHlkb8w
  ✗ True: cook         Pred: add          | [MASK] the bacon on a grill for one minute
  ✗ True: add          Pred: place        | [MASK] shredded cheese onto the bread
  ✗ True: add          Pred: place        | [MASK] tomatoes to the bread slices
  ✗ True: add          Pred: place        | [MASK] the bacon to the bread slices
  ✗ True: add          Pred: place        | [MASK] basil to the bread slices
  ✗ True: add          Pred: place        | [MASK] spinach to the bread slices
  ✗ True: top          Pred: place        | [MASK] the sandwiches with bread
  ✗ True: cook         Pred: place        | [MASK] the sandwiches on the grill

Video: -ErPSunMfcs
  ✗ True: prepare      Pred: add          | [MASK] some chicken cutlets and immense them in two beat eggs
  ✗ True: panko        Pred: place        | coat the chicken cutlets with mixture of grated parmesan cheese and [MASK] breadcrumbs
  ✗ True: coat         Pred: add          | press chicken cutlets hard to [MASK] 

In [25]:
# Detailed per-segment evaluation
model.eval()
records = []

with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc="Detailed eval"):
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        labels = labels.to(DEVICE)
        logits = model(**inputs).logits
        top5   = logits.topk(5, dim=-1).indices

        for i in range(len(labels)):
            true_label = labels[i].item()
            top5_list  = top5[i].tolist()
            records.append({
                'video_id'    : df_val.iloc[len(records)]['video_id'],
                'segment_id'  : df_val.iloc[len(records)]['segment_id'],
                'ground_truth': id2label[true_label],
                'top1_pred'   : id2label[logits[i].argmax(-1).item()],
                'top1_correct': logits[i].argmax(-1).item() == true_label,
                'top5_correct': true_label in top5_list,
            })

results_df = pd.DataFrame(records)

# Now check error clustering
wrong = results_df[results_df['top1_correct'] == False]
print(wrong['video_id'].value_counts().head(10))

Detailed eval:   0%|          | 0/817 [00:00<?, ?it/s]

[h264 @ 0x273b0f00] mmco: unref short failure
[h264 @ 0x273b0f00] mmco: unref short failure
[h264 @ 0x50e96880] mmco: unref short failure
[h264 @ 0x50e96880] mmco: unref short failure
[h264 @ 0x455f4a80] mmco: unref short failure
[h264 @ 0x455f4a80] mmco: unref short failure
[h264 @ 0x4a5be0c0] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x4a5be0c0] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x4a5be0c0] mmco: unref short failure
[h264 @ 0x4a5be0c0] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x4a5be0c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x273c4c40] mmco: unref sh

KeyboardInterrupt: 

In [22]:
# Inspect predictions for the first batch
model.eval()
inputs, labels = next(iter(val_loader))
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits

for i in range(len(labels)):
    true_verb = id2label[labels[i].item()]
    pred_verb = id2label[logits[i].argmax(-1).item()]
    top5_verbs = [id2label[j] for j in logits[i].topk(5).indices.tolist()]
    print(f"True: {true_verb:15s} | Pred: {pred_verb:15s} | Top5: {top5_verbs}")

[h264 @ 0x243c1580] mmco: unref short failure
[h264 @ 0x243c1580] mmco: unref short failure
[h264 @ 0x5013e700] mmco: unref short failure
[h264 @ 0x5013e700] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x455f4a80] mmco: unref short failure


True: combine         | Pred: add             | Top5: ['add', 'mix', 'combine', 'pour', 'put']
True: place           | Pred: roll            | Top5: ['roll', 'fold', 'knead', 'wrap', 'spread']
True: place           | Pred: place           | Top5: ['place', 'add', 'put', 'heat', 'take']
True: mix             | Pred: add             | Top5: ['add', 'mix', 'boil', 'stir', 'cook']


In [ ]:
# Inspect predictions for a specific batch with video context
model.eval()

# grab any batch from val_loader — change the number to skip batches
batch_num = 5
val_iter  = iter(val_loader)
for _ in range(batch_num):
    inputs, labels = next(val_iter)

inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits

# match back to df_val rows by scanning for matching verb labels
# (approximate — finds first df_val row with matching verb label)
print(f"{'Video ID':<20} {'True Verb':<15} {'Top-1 Pred':<15} {'Top-5 Predictions':<50} {'Caption'}")
print("-" * 130)

for i in range(len(labels)):
    true_label = labels[i].item()
    true_verb  = id2label[true_label]
    pred_verb  = id2label[logits[i].argmax(-1).item()]
    top5_verbs = [id2label[j] for j in logits[i].topk(5).indices.tolist()]

    # look up the caption and video_id from df_val
    match = df_val[df_val['verb_label'] == true_label].iloc[0]
    video_id = match['video_id']
    caption  = match['masked_caption']

    correct = "✓" if pred_verb == true_verb else "✗"
    print(f"{video_id:<20} {true_verb:<15} {correct} {pred_verb:<13} {str(top5_verbs):<50} {caption}")

In [29]:
# ── Final evaluation — video-only V-JEPA2 verb classification ──────────────
model.eval()
records = []

with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc="Evaluating"):
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        labels = labels.to(DEVICE)
        logits = model(**inputs).logits          # (B, n_verbs)
        top5   = logits.topk(5, dim=-1).indices  # (B, 5)

        for i in range(len(labels)):
            true_label = labels[i].item()
            top5_list  = top5[i].tolist()
            records.append({
                'top1_correct': logits[i].argmax(-1).item() == true_label,
                'top5_correct': true_label in top5_list,
                'ground_truth': id2label[true_label],
                'top1_pred'   : id2label[logits[i].argmax(-1).item()],
                'top5_preds'  : [id2label[j] for j in top5_list],
            })

results_df = pd.DataFrame(records)

# Add segment/video IDs — safe alignment via reset index
results_df['segment_id'] = df_val['segment_id'].values[:len(results_df)]
results_df['video_id']   = df_val['video_id'].values[:len(results_df)]

# ── Summary ─────────────────────────────────────────────────────────────────
top1 = results_df['top1_correct'].mean()
top5 = results_df['top5_correct'].mean()
print(f"Validation Top-1 : {top1:.4f}  ({top1*100:.2f}%)")
print(f"Validation Top-5 : {top5:.4f}  ({top5*100:.2f}%)")

# ── Per-verb breakdown ───────────────────────────────────────────────────────
per_verb = results_df.groupby('ground_truth').agg(
    count    = ('top1_correct', 'count'),
    top1_acc = ('top1_correct', 'mean'),
    top5_acc = ('top5_correct', 'mean'),
).sort_values('count', ascending=False)
print("\nTop 20 most frequent verbs:")
print(per_verb.head(20).to_string())

# ── Save ─────────────────────────────────────────────────────────────────────
out_path = os.path.join(PATHS['processed'], 'nb5_vjepa2_videonly_results.csv')
results_df.to_csv(out_path, index=False)
print(f"\nResults saved → {out_path}")

Evaluating:   0%|          | 0/817 [00:00<?, ?it/s]

[h264 @ 0x501e2380] mmco: unref short failure
[h264 @ 0x501e2380] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x50001a00] mmco: unref short failure
[h264 @ 0x50001a00] mmco: unref short failure
[h264 @ 0x4a5be0c0] mmco: unref short failure
[h264 @ 0x51170500] mmco: unref short failure
[h264 @ 0x5203db00] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x243b52c0] mmco: unref short failure
[h264 @ 0x4a5be0c0] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x51450440] mmco: unref short failure
[h264 @ 0x51450440] mmco: unref short failure
[h264 @ 0x4a5be0c0] mmco: unref short failure
[h264 @ 0x4a5be0c0] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x23cc4500] mmco: unref short failure
[h264 @ 0x273e6ec0] mmco: unref short failure
[h264 @ 0x455f4a80] mmco: unref sh

Validation Top-1 : 0.4287  (42.87%)
Validation Top-5 : 0.7188  (71.88%)

Top 20 most frequent verbs:
              count  top1_acc  top5_acc
ground_truth                           
add            1165  0.913305  0.991416
mix             162  0.154321  0.870370
place           156  0.397436  0.788462
cut             143  0.762238  0.909091
pour            132  0.121212  0.818182
cook            111  0.153153  0.756757
chop             95  0.294737  0.800000
put              90  0.000000  0.700000
remove           55  0.072727  0.690909
spread           53  0.188679  0.660377
heat             53  0.018868  0.773585
cover            51  0.156863  0.490196
fry              49  0.265306  0.795918
take             45  0.000000  0.200000
slice            42  0.000000  0.952381
roll             40  0.625000  0.825000
stir             40  0.000000  0.675000
sprinkle         40  0.125000  0.725000
boil             32  0.156250  0.718750
combine          30  0.000000  0.533333

Results saved → /h

## Save verb vocabulary

Save label mappings so downstream notebooks can load them.

In [24]:
vocab_path = os.path.join(PATHS['processed'], 'verb_vocab.json')
with open(vocab_path, 'w') as f:
    json.dump({
        'label2id': label2id,
        'id2label': {str(k): v for k, v in id2label.items()}
    }, f, indent=2)
print(f"Verb vocabulary saved -> {vocab_path}")
print(f"Total verbs: {n_verbs}")

Verb vocabulary saved -> /home/ubuntu/youcook2-videos/processed/verb_vocab.json
Total verbs: 282


Part 2 --> generating embeddings using the trained pooler + frozen backbone to align with dinotxt
--> we need to get the embeddings from the pipeline within the classification wrapper by using a hook

In [32]:
# Run this once before the extraction loop to see what the model returns
clip, _ = all_ds[0]
inputs  = processor([clip], return_tensors='pt')
inputs  = {k: v.to(DEVICE) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    
print(type(outputs))
print(dir(outputs))

<class 'transformers.modeling_outputs.ImageClassifierOutput'>
['__annotations__', '__class__', '__class_getitem__', '__contains__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__ior__', '__iter__', '__le__', '__len__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__or__', '__post_init__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__ror__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', 'attentions', 'clear', 'copy', 'fromkeys', 'get', 'hidden_states', 'items', 'keys', 'logits', 'loss', 'move_to_end', 'pop', 'popitem', 'setdefault', 'to_tuple', 'update', 'values']


In [33]:
# Run this once to see what model.classifier expects
print(model.classifier)

Linear(in_features=1024, out_features=282, bias=True)


In [36]:
# Sanity check — verify the hook captures a valid embedding before running full extraction
model.eval()

embedding_store = {}
def hook_fn(module, input, output):
    embedding_store['embedding'] = input[0].detach().cpu()

hook = model.classifier.register_forward_hook(hook_fn)

with torch.no_grad():
    clip, label = all_ds[0]
    inputs      = processor([clip], return_tensors='pt')
    inputs      = {k: v.to(DEVICE) for k, v in inputs.items()}
    outputs     = model(**inputs)

hook.remove()

# Check 1 — did the hook fire at all?
assert 'embedding' in embedding_store, "Hook did not fire — embedding_store is empty"

emb = embedding_store['embedding'].squeeze(0)

# Check 2 — is the shape right?
assert emb.shape == (1024,), f"Expected (1024,) but got {emb.shape}"

# Check 3 — is it non-zero? (all zeros would mean something went wrong)
assert emb.norm().item() > 0, "Embedding is all zeros — something went wrong"

# Check 4 — does it match what the classifier expects?
assert emb.shape[0] == model.classifier.in_features, \
    f"Embedding dim {emb.shape[0]} doesn't match classifier input {model.classifier.in_features}"

print(f"Hook fired       : ✓")
print(f"Embedding shape  : {emb.shape}  ✓")
print(f"L2 norm          : {emb.norm():.4f}  ✓")
print(f"Matches classifier input dim : {emb.shape[0]} == {model.classifier.in_features}  ✓")
print(f"Ground truth verb: {id2label[label]}")
print(f"\nAll checks passed — safe to run full extraction")

Hook fired       : ✓
Embedding shape  : torch.Size([1024])  ✓
L2 norm          : 46.0582  ✓
Matches classifier input dim : 1024 == 1024  ✓
Ground truth verb: crush

All checks passed — safe to run full extraction


In [37]:
# Extract and save V-JEPA2 pooled embeddings for every segment
# These are the post-pooler representations — the same features the classifier trained on
# Saved as {segment_id}.pt so the DinoTXT notebook can load them by segment ID

import os

EMB_DIR = os.path.join(PATHS['embeddings'], 'vjepa2')
os.makedirs(EMB_DIR, exist_ok=True)

model.eval()

# We need segment_ids alongside clips — iterate over full df directly
all_ds = YouCookVerbDataset(df, PATHS['videos'], FRAMES_PER_CLIP)

# Hook to capture the input to the classifier (i.e. the pooled embedding)
embedding_store = {}

def hook_fn(module, input, output):
    embedding_store['embedding'] = input[0].detach().cpu()

hook    = model.classifier.register_forward_hook(hook_fn)
skipped = 0

with torch.no_grad():
    for idx in tqdm(range(len(all_ds)), desc="Extracting embeddings"):
        segment_id = df.iloc[idx]['segment_id']
        emb_path   = os.path.join(EMB_DIR, f"{segment_id}.pt")

        # Skip if already saved
        if os.path.exists(emb_path):
            continue

        try:
            clip, _ = all_ds[idx]
            inputs  = processor([clip], return_tensors='pt')
            inputs  = {k: v.to(DEVICE) for k, v in inputs.items()}

            # Forward pass triggers the hook
            model(**inputs)

            # embedding_store now has the (1024,) pooled vector
            embedding = embedding_store['embedding'].squeeze(0)
            torch.save(embedding, emb_path)

        except Exception as e:
            print(f"Skipped {segment_id}: {e}")
            skipped += 1

# Always remove the hook when done
hook.remove()

print(f"Done. Embeddings saved to {EMB_DIR}")
print(f"Total saved : {len(os.listdir(EMB_DIR)):,}")
print(f"Skipped     : {skipped}")

Extracting embeddings:   0%|          | 0/13003 [00:00<?, ?it/s]

[h264 @ 0x51053500] mmco: unref short failure
[h264 @ 0x51053500] mmco: unref short failure
[h264 @ 0x5103eb80] mmco: unref short failure
[h264 @ 0x51d8bb00] mmco: unref short failure
[h264 @ 0x50405240] mmco: unref short failure
[h264 @ 0x50405240] mmco: unref short failure
[h264 @ 0x50dffc40] mmco: unref short failure
[h264 @ 0x50dffc40] mmco: unref short failure
[h264 @ 0x50ba24c0] mmco: unref short failure
[h264 @ 0x50ba24c0] mmco: unref short failure
[h264 @ 0x52a51b80] mmco: unref short failure
[h264 @ 0x52a51b80] mmco: unref short failure
[h264 @ 0x52a51b80] mmco: unref short failure
[h264 @ 0x521ab6c0] mmco: unref short failure
[h264 @ 0x521ab6c0] mmco: unref short failure
[h264 @ 0x502c4880] mmco: unref short failure
[h264 @ 0x50ba24c0] mmco: unref short failure
[h264 @ 0x50ba24c0] mmco: unref short failure
[h264 @ 0x50ba24c0] mmco: unref short failure
[h264 @ 0x50ba24c0] mmco: unref short failure
[h264 @ 0x50ba24c0] mmco: unref short failure
[h264 @ 0x52b16500] mmco: unref sh

Done. Embeddings saved to /home/ubuntu/youcook2-videos/embeddings/dinov2/vjepa2
Total saved : 13,003
Skipped     : 0


In [ ]:
import sys

DINOV2_REPO = os.path.join(PATHS['repos'], 'dinov2')
if DINOV2_REPO not in sys.path:
    sys.path.insert(0, DINOV2_REPO)

from dinov2.hub.dinotxt import dinov2_vitl14_reg4_dinotxt_tet1280d20h24l, get_tokenizer
import torch.nn as nn

# Load frozen DinoTXT text tower
print("Loading DinoTXT...")
dinotxt   = dinov2_vitl14_reg4_dinotxt_tet1280d20h24l().to(DEVICE)
tokenizer = get_tokenizer()
dinotxt.eval()
for p in dinotxt.parameters():
    p.requires_grad = False

print("DinoTXT loaded and frozen ✓")

# Learnable projection: maps V-JEPA2 (1024) → DinoTXT text space (1280)
# Following V-JEPA2 paper Section 7.1 — visual instruction tuning via a learnable projector MLP
VJEPA2_DIM  = model.classifier.in_features  # 1024, read directly from trained model
test_tokens = tokenizer.tokenize(["test caption"]).to(DEVICE)
with torch.no_grad():
    test_out = dinotxt.encode_text(test_tokens)
DINOTXT_DIM = test_out.shape[-1]
print(f"DinoTXT output dim : {DINOTXT_DIM}")

projection = nn.Linear(VJEPA2_DIM, DINOTXT_DIM, bias=False).to(DEVICE)
optimizer  = torch.optim.AdamW(projection.parameters(), lr=1e-4)

print(f"Projection layer : Linear({VJEPA2_DIM} → {DINOTXT_DIM})")
print(f"Trainable params : {sum(p.numel() for p in projection.parameters()):,}")

Loading DinoTXT...


/home/ubuntu/youcook2-videos/repos/dinov2/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/ubuntu/youcook2-videos/repos/dinov2/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/ubuntu/youcook2-videos/repos/dinov2/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


DinoTXT loaded and frozen ✓
DinoTXT output dim : 2048
Projection layer : Linear(1024 → 2048)
Trainable params : 2,097,152


In [40]:
# Build verb vocabulary — same as DinoTXT notebook
verb_vocab = sorted(df['extracted_verb'].unique().tolist())
n_verbs    = len(verb_vocab)
print(f"Verb classes : {n_verbs}")

# Get context length empirically from the tokenizer
test_tokens    = tokenizer.tokenize(["test"]).to(DEVICE)
CONTEXT_LENGTH = test_tokens.shape[-1]
print(f"Context length : {CONTEXT_LENGTH}")

@torch.no_grad()
def get_text_embeddings_for_segment(masked_caption: str) -> torch.Tensor:
    """
    Replace [MASK] with every verb in vocab and encode all candidates.
    Returns (n_verbs, DINOTXT_DIM), L2-normalised.
    """
    caption_context = masked_caption.replace('[MASK]', '{}')
    verb_texts      = [caption_context.format(v) for v in verb_vocab]
    tokens          = tokenizer.tokenize(verb_texts).to(DEVICE)
    text_features   = dinotxt.encode_text(tokens)           # (n_verbs, 1280)
    text_features   = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features

# Sanity check
test_emb = get_text_embeddings_for_segment(df['masked_caption'].iloc[0])
print(f"Text embeddings shape : {test_emb.shape}  (expected: ({n_verbs}, {DINOTXT_DIM}))")
print(f"L2 norm               : {test_emb.norm(dim=-1).mean():.4f}  (expected: ~1.0)")

Verb classes : 282
Context length : 77
Text embeddings shape : torch.Size([282, 2048])  (expected: (282, 2048))
L2 norm               : 1.0000  (expected: ~1.0)


In [ ]:
# Train the projection to align V-JEPA2 embeddings with DinoTXT text space
# Loss: cosine similarity between projected video embedding and correct verb text embedding

EMB_DIR     = os.path.join(PATHS['embeddings'], 'vjepa2')
PROJ_EPOCHS = 5

df_train_proj = df[df['subset'] == 'training'].reset_index(drop=True)

print(f"Training projection layer — {PROJ_EPOCHS} epochs")

for epoch in range(1, PROJ_EPOCHS + 1):
    projection.train()
    total_loss    = 0.0
    total_correct = 0
    total_samples = 0

    for _, row in tqdm(df_train_proj.iterrows(), total=len(df_train_proj),
                       desc=f"Epoch {epoch}/{PROJ_EPOCHS}"):
        emb_path = os.path.join(EMB_DIR, f"{row['segment_id']}.pt")
        if not os.path.exists(emb_path):
            continue

        # Load saved V-JEPA2 embedding and project into DinoTXT space
        video_emb = torch.load(emb_path, map_location=DEVICE)   # (1024,)
        video_emb = projection(video_emb)                        # (1280,)
        video_emb = video_emb / video_emb.norm()                 # L2 normalise

        # Get text embeddings for all verbs in this caption's context
        text_embs  = get_text_embeddings_for_segment(row['masked_caption'])  # (n_verbs, 1280)

        # Cosine similarity → (n_verbs,)
        similarity = text_embs @ video_emb

        # Ground truth index
        ground_truth = row['extracted_verb']
        if ground_truth not in verb_vocab:
            continue
        gt_idx = verb_vocab.index(ground_truth)

        # CrossEntropy loss — push similarity of correct verb to be highest
        loss = nn.functional.cross_entropy(
            similarity.unsqueeze(0),
            torch.tensor([gt_idx], device=DEVICE)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss    += loss.item()
        total_correct += (similarity.argmax().item() == gt_idx)
        total_samples += 1

    print(f"Epoch {epoch} | Loss: {total_loss/total_samples:.4f} "
          f"| Train Top-1: {total_correct/total_samples:.4f}")

print("Projection training complete.")

Precomputing text embeddings...
Total unique captions : 12,030


Encoding captions:   0%|          | 0/1504 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ProjectionDataset(Dataset):
    def __init__(self, df, emb_dir):
        # Only keep rows where embedding exists
        self.records = []
        for _, row in df.iterrows():
            emb_path = os.path.join(emb_dir, f"{row['segment_id']}.pt")
            if os.path.exists(emb_path) and row['extracted_verb'] in verb_vocab:
                self.records.append({
                    'emb_path'      : emb_path,
                    'masked_caption': row['masked_caption'],
                    'gt_idx'        : verb_vocab.index(row['extracted_verb']),
                })

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r         = self.records[idx]
        video_emb = torch.load(r['emb_path'], map_location='cpu')  # (1024,)
        return video_emb, r['masked_caption'], r['gt_idx']


def proj_collate(batch):
    video_embs, captions, gt_idxs = zip(*batch)
    return (
        torch.stack(video_embs),               # (B, 1024)
        list(captions),
        torch.tensor(gt_idxs, dtype=torch.long)
    )


EMB_DIR        = os.path.join(PATHS['embeddings'], 'vjepa2')
PROJ_EPOCHS    = 5
PROJ_BATCH     = 32

train_proj_ds  = ProjectionDataset(df[df['subset'] == 'training'].reset_index(drop=True), EMB_DIR)
train_proj_dl  = DataLoader(train_proj_ds, batch_size=PROJ_BATCH, shuffle=True,
                            collate_fn=proj_collate, num_workers=4, pin_memory=True)

print(f"Training projection — {PROJ_EPOCHS} epochs, batch={PROJ_BATCH}")
print(f"Batches per epoch : {len(train_proj_dl):,}")

for epoch in range(1, PROJ_EPOCHS + 1):
    projection.train()
    total_loss    = 0.0
    total_correct = 0
    total_samples = 0

    for video_embs, captions, gt_idxs in tqdm(train_proj_dl,
                                               desc=f"Epoch {epoch}/{PROJ_EPOCHS}"):
        video_embs = video_embs.to(DEVICE)   # (B, 1024)
        gt_idxs    = gt_idxs.to(DEVICE)      # (B,)

        # Project entire batch at once
        video_embs = projection(video_embs)                               # (B, 2048)
        video_embs = video_embs / video_embs.norm(dim=-1, keepdim=True)  # L2 normalise

        # Encode text for each caption in the batch — (B, n_verbs, 2048)
        text_embs = torch.stack([
            caption_text_embs[cap].to(DEVICE) for cap in captions
        ])  # (B, n_verbs, 2048)

        # Cosine similarity for each sample: (B, n_verbs)
        # bmm: (B, n_verbs, 2048) @ (B, 2048, 1) → (B, n_verbs, 1) → (B, n_verbs)
        similarity = torch.bmm(text_embs, video_embs.unsqueeze(-1)).squeeze(-1)

        # CrossEntropy over verb similarities
        loss = nn.functional.cross_entropy(similarity, gt_idxs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss    += loss.item() * len(gt_idxs)
        total_correct += (similarity.argmax(dim=-1) == gt_idxs).sum().item()
        total_samples += len(gt_idxs)

    print(f"Epoch {epoch} | Loss: {total_loss/total_samples:.4f} "
          f"| Train Top-1: {total_correct/total_samples:.4f}")

print("Projection training complete.")

Training projection — 5 epochs, batch=32
Batches per epoch : 305


Epoch 1/5:   0%|          | 0/305 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Zero-shot verb prediction using projected V-JEPA2 embeddings + DinoTXT text tower
# Only change from original DinoTXT notebook: load from vjepa2/ and apply projection

df_val_proj = df[df['subset'] == 'validation'].reset_index(drop=True)

projection.eval()
results = []
missing = 0

with torch.no_grad():
    for _, row in tqdm(df_val_proj.iterrows(), total=len(df_val_proj),
                       desc="Zero-shot prediction"):
        emb_path = os.path.join(EMB_DIR, f"{row['segment_id']}.pt")

        if not os.path.exists(emb_path):
            missing += 1
            continue

        # Load V-JEPA2 embedding, project into DinoTXT space
        video_emb = torch.load(emb_path, map_location=DEVICE)  # (1024,)
        video_emb = projection(video_emb)                       # (1280,)
        video_emb = video_emb / video_emb.norm()                # L2 normalise

        # Encode all verbs in this caption's context — identical to DinoTXT notebook
        text_embs  = get_text_embeddings_for_segment(row['masked_caption'])  # (n_verbs, 1280)
        similarity = text_embs @ video_emb

        top5_indices = similarity.topk(5).indices.tolist()
        top1_verb    = verb_vocab[top5_indices[0]]
        ground_truth = row['extracted_verb']
        gt_idx       = verb_vocab.index(ground_truth) if ground_truth in verb_vocab else -1

        results.append({
            'segment_id'    : row['segment_id'],
            'subset'        : row['subset'],
            'ground_truth'  : ground_truth,
            'top1_pred'     : top1_verb,
            'top5_preds'    : [verb_vocab[i] for i in top5_indices],
            'top1_correct'  : top1_verb == ground_truth,
            'top5_correct'  : gt_idx in top5_indices,
            'masked_caption': row['masked_caption'],
        })

print(f"Predicted          : {len(results):,}")
print(f"Missing embeddings : {missing:,}")